# Mô hình ALS

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
import datetime

# Khởi tạo Spark với cấu hình giới hạn RAM để tránh sập Driver
spark = SparkSession.builder \
    .appName("ALS_Professional_Training") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

# Thiết lập thư mục Checkpoint để giải phóng RAM trong quá trình lặp
spark.sparkContext.setCheckpointDir("s3a://gold/checkpoints")


In [3]:
# Tải dữ liệu từ tầng Gold
events_df = spark.table("gold_db.fact_events")

# Định nghĩa hàm tính Interaction Score (có Log-scaling và Min-Max để hạ RMSE)
def get_interaction_matrix(df, ref_date):
    raw_matrix = df.groupBy("user_id", "product_id").agg(
        F.sum(F.when(F.col("event_type") == "view", 1)
              .when(F.col("event_type") == "cart", 3)
              .when(F.col("event_type") == "purchase", 5)
              .otherwise(0)).alias("base_weight"),
        F.count("event_type").alias("interaction_count"),
        F.min(F.datediff(F.lit(ref_date), F.col("event_time"))).alias("days_since_last_interact")
    ).withColumn("raw_score", 
        (F.col("base_weight") * F.log1p(F.col("interaction_count"))) / 
        (F.lit(1) + F.lit(0.01) * F.col("days_since_last_interact")))

    # Log transformation và Min-Max về [1, 10]
    log_df = raw_matrix.withColumn("log_score", F.log1p("raw_score"))
    stats = log_df.select(F.min("log_score").alias("min_s"), F.max("log_score").alias("max_s")).first()
    
    return log_df.withColumn("interaction_score", 
        F.round(((F.col("log_score") - stats['min_s']) / (stats['max_s'] - stats['min_s'])) * 9 + 1, 4)
    ).select("user_id", "product_id", "interaction_score")

# Chia tập theo thời gian
max_date = events_df.select(F.max("event_time")).first()[0]
cutoff = max_date - datetime.timedelta(days=30)

train_events = events_df.filter(F.col("event_time") < cutoff)
test_events = events_df.filter(F.col("event_time") >= cutoff)

train_matrix = get_interaction_matrix(train_events, cutoff)
test_matrix = get_interaction_matrix(test_events, max_date)

# Indexing IDs (ALS cần ID dạng số nguyên)
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_index", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_index", handleInvalid="skip")

indexer_model = Pipeline(stages=[user_indexer, item_indexer]).fit(train_matrix)
train_ready = indexer_model.transform(train_matrix).cache()
test_ready = indexer_model.transform(test_matrix).cache()

print(f"Train size: {train_ready.count()} | Test size: {test_ready.count()}")

Train size: 447752 | Test size: 7125


In [3]:
# Khởi tạo Model ALS Implicit
als = ALS(
    userCol="user_index", 
    itemCol="item_index", 
    ratingCol="interaction_score",
    coldStartStrategy="drop",
    nonnegative=True,
    implicitPrefs=True, # Dữ liệu hành vi nên dùng Implicit
    maxIter=15,
    checkpointInterval=2
)

# Lưới tham số  
param_grid = ParamGridBuilder() \
    .addGrid(als.rank, [10, 20, 40]) \
    .addGrid(als.regParam, [0.01, 0.1]) \
    .addGrid(als.alpha, [1.0, 40.0]) \
    .build()

evaluator = RegressionEvaluator(metricName="rmse", labelCol="interaction_score", predictionCol="prediction")

# CrossValidator chạy tuần tự (parallelism=1) để an toàn cho RAM
cv = CrossValidator(estimator=als, estimatorParamMaps=param_grid, evaluator=evaluator, numFolds=2, parallelism=1)

cv_model = cv.fit(train_ready)
best_model = cv_model.bestModel

print(f"Best Rank: {best_model.rank}")
print(f"Best RegParam: {best_model._java_obj.parent().getRegParam()}")
print(f"Best Alpha: {best_model._java_obj.parent().getAlpha()}")

Best Rank: 40
Best RegParam: 0.1
Best Alpha: 40.0


In [4]:
import time
from pyspark.sql.window import Window

#  NHẬP THAM SỐ TỐI ƯU TẠI ĐÂY
BEST_RANK = 40
BEST_REG = 0.1
BEST_ALPHA = 40.0
K_TOP = 10  # Số lượng gợi ý (Top 10)

# TÁI TẠO BỘ ÁNH XẠ ID (Indexer) 
# Đảm bảo có user_map và item_map để dịch ngược ID ở bước cuối
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_index", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_index", handleInvalid="skip")

indexer_pipeline = Pipeline(stages=[user_indexer, item_indexer])
indexer_model = indexer_pipeline.fit(train_matrix)

train_ready = indexer_model.transform(train_matrix).cache()
test_ready = indexer_model.transform(test_matrix).cache()

user_map = train_ready.select("user_id", "user_index").distinct()
item_map = train_ready.select("product_id", "item_index").distinct()

# HUẤN LUYỆN MÔ HÌNH VÀ ĐO THỜI GIAN 
als_final = ALS(
    userCol="user_index", 
    itemCol="item_index", 
    ratingCol="interaction_score",
    rank=BEST_RANK,
    regParam=BEST_REG,
    alpha=BEST_ALPHA,
    implicitPrefs=True,
    coldStartStrategy="drop",
    nonnegative=True,
    maxIter=15,
    checkpointInterval=2
)

start_time = time.time()
final_model = als_final.fit(train_ready) 
training_time = time.time() - start_time

# TÍNH RMSE VÀ MAPE
predictions = final_model.transform(test_ready).filter("prediction IS NOT NULL")

evaluator = RegressionEvaluator(metricName="rmse", labelCol="interaction_score", predictionCol="prediction")
rmse_val = evaluator.evaluate(predictions)

mape_val = predictions.withColumn("abs_perc_err", 
    F.abs(F.col("interaction_score") - F.col("prediction")) / F.col("interaction_score")) \
    .select(F.avg("abs_perc_err")).first()[0] * 100

# TÍNH PRECISION, RECALL, F1 
user_subset = test_ready.select("user_index").distinct()
user_recs = final_model.recommendForUserSubset(user_subset, K_TOP)

actual_items = test_ready.groupBy("user_index").agg(F.collect_set("item_index").alias("actual_list"))
predicted_items = user_recs.select("user_index", F.col("recommendations.item_index").alias("pred_list"))
comparison = actual_items.join(predicted_items, "user_index")

match_df = comparison.withColumn("hits", F.size(F.array_intersect("actual_list", "pred_list")))
precision_at_k = match_df.select(F.avg(F.col("hits") / K_TOP)).first()[0]
recall_at_k = match_df.select(F.avg(F.col("hits") / F.size("actual_list"))).first()[0]
f1_val = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k) if (precision_at_k + recall_at_k) > 0 else 0

#  IN KẾT QUẢ TỔNG HỢP  
print("\n" + "="*85)
print(f"{'Mô hình':<15} | {'MAPE':<10} | {'RMSE':<10} | {f'Prec@{K_TOP}':<10} | {f'Rec@{K_TOP}':<10} | {'F1':<10} | {'Time':<10}")
print("-" * 85)
print(f"{'ALS (Final)':<15} | {mape_val:>8.2f}% | {rmse_val:>10.4f} | {precision_at_k:>10.4f} | {recall_at_k:>10.4f} | {f1_val:>10.4f} | {training_time:>8.2f}s")
print("="*85 + "\n")

# LƯU KẾT QUẢ GỢI Ý VÀO TẦNG GOLD   
all_user_recs = final_model.recommendForAllUsers(K_TOP)
recs_exploded = all_user_recs.withColumn("rec", F.explode("recommendations")) \
    .select("user_index", F.col("rec.item_index").alias("item_index"), F.col("rec.rating").alias("score"))

final_als_recommendations = recs_exploded.join(user_map, on="user_index") \
    .join(item_map, on="item_index") \
    .select("user_id", "product_id", "score")

final_als_recommendations.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", "s3a://gold/recommendations_als") \
    .saveAsTable("gold_db.recommendations_als")

print(f"SUCCESS: Đã lưu gợi ý cho {all_user_recs.count()} người dùng.")


Mô hình         | MAPE       | RMSE       | Prec@10    | Rec@10     | F1         | Time      
-------------------------------------------------------------------------------------
ALS (Final)     |    80.77% |     1.9949 |     0.0355 |     0.2381 |     0.0618 |   411.09s

SUCCESS: Đã lưu gợi ý cho 331649 người dùng.


# Mô hình Popularity (Baseline)
Dùng để so sánh với việc có và không có sử dụng các mô hình ALS, CB, IB, Hybird


In [5]:
from pyspark.sql import functions as F

# Lấy danh sách Top 10 sản phẩm bán chạy nhất từ tầng Gold
# Đây là danh sách gợi ý giống hệt nhau cho mọi user
top_10_trending = spark.table("gold_db.top_trending") \
    .select("product_id") \
    .limit(10) \
    .agg(F.collect_set("product_id").alias("pred_list")) \
    .first()["pred_list"]

print(f"Danh sách gợi ý (top 10 sản phẩm bán chạy): {top_10_trending}")

# Lấy dữ liệu thực tế từ tập Test 
actual_items = test_matrix.groupBy("user_id").agg(F.collect_set("product_id").alias("actual_list"))

# Tính toán Precision và Recall cho Popularity
# Vì mọi user đều nhận được cùng 1 danh sách, ta dùng F.lit()
pop_comparison = actual_items.withColumn("pred_list", F.array([F.lit(x) for x in top_10_trending]))

# Tính Hits (số món trùng khớp)
pop_match = pop_comparison.withColumn("hits", F.size(F.array_intersect("actual_list", "pred_list")))

# Tính chỉ số trung bình
pop_precision = pop_match.select(F.avg(F.col("hits") / 10)).first()[0]
pop_recall = pop_match.select(F.avg(F.col("hits") / F.size("actual_list"))).first()[0]
pop_f1 = 2 * (pop_precision * pop_recall) / (pop_precision + pop_recall) if (pop_precision + pop_recall) > 0 else 0

# IN KẾT QUẢ 
print("\n" + "="*50)
print(f"KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH POPULARITY (BASELINE):")
print(f"   - Precision@10: {pop_precision:.4f}")
print(f"   - Recall@10: {pop_recall:.4f}")
print(f"   - F1-Score: {pop_f1:.4f}")
print(f"   - Thời gian: 0.01s (Vì chỉ là lấy bảng có sẵn)")
print("="*50)

Danh sách gợi ý (top 10 sản phẩm bán chạy): [877060, 914954, 1843507, 1795171, 1341995, 4079422, 4183872, 4154618, 846904, 523117]

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH POPULARITY (BASELINE):
   - Precision@10: 0.0013
   - Recall@10: 0.0065
   - F1-Score: 0.0021
   - Thời gian: 0.01s (Vì chỉ là lấy bảng có sẵn)
